# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 제출 정보

- 이름: 권혁수
- GitHub ID: kwonhyeoksoo
- 작성일: 2026-10-14
- 최종 제출 URL: https://github.com/kwonhyeoksoo/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb

## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-study\llm-data-analysis-course\notebooks
프로젝트 루트: c:\dev\llm-data-analysis-study\llm-data-analysis-course
원본 데이터 폴더: c:\dev\llm-data-analysis-study\llm-data-analysis-course\practice\chapter05\data\raw
전처리 데이터 폴더: c:\dev\llm-data-analysis-study\llm-data-analysis-course\practice\chapter05\data\processed
보고서 폴더: c:\dev\llm-data-analysis-study\llm-data-analysis-course\practice\chapter05\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [3]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [4]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [5]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [6]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [7]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
age,4,2.63
city,2,1.32
name,0,0.00
customer_id,0,0.00
gender,0,0.00
signup_date,0,0.00


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [8]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [9]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [10]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


In [11]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 1. 전처리 전 상태 기록

- 각 데이터 shape: customers (152, 6) / products (102, 4) / orders (302, 5) / order_items (765, 5)
- 결측치: customers.age 4건(2.63%), customers.city 2건(1.32%); 그 외 모든 컬럼 0건
- 전체 중복: 4개 데이터셋 모두 완전 중복 행 2건씩
- 주요 ID 중복: customer_id 2건, product_id 2건, order_id 2건, order_item_id 2건 (전체 행 중복 수와 정확히 일치하여 완전히 동일한 행이 그대로 중복 삽입된 것으로 판단)
- 타입 문제 후보: 아직 dtype을 직접 확인하지 않았으나, price/quantity/unit_price가 문자열(쉼표 포함 등)로 저장되어 있을 가능성이 있어 이후 단계에서 확인 예정

![전처리 전 상태](images/step01_before.png)

### 결과 관찰

4개 원본 데이터셋 모두 결측치는 customers에만 존재했다(age 4건, city 2건). 반면 전체 행 중복과 주요 ID 중복은 4개 데이터셋 모두에서 정확히 2건씩 발생했으며, 두 수치가 완전히 일치한다는 점에서 이는 서로 다른 레코드의 ID가 우연히 겹친 것이 아니라, 동일한 행이 그대로 두 번 입력된 중복으로 판단했다.

### 나의 해석과 판단

먼저 처리해야 할 문제는 전체 행 중복이라고 판단했다. 중복 행을 그대로 두고 분석하면 매출이나 건수가 실제보다 부풀려지기 때문에 다른 어떤 처리보다 먼저 제거해야 한다고 생각했다. 결측치(age, city)는 중복보다는 영향 범위가 작지만, 삭제가 아니라 값을 대체하는 방식(중앙값, Unknown)을 선택했는데, 이는 고객 데이터를 통째로 버리면 다른 분석(주문, 매출 등)에서 해당 고객 정보가 통째로 사라지기 때문이다.

### 업무·분석적 의미

전처리 전 상태를 먼저 숫자로 기록해두면 이후 처리 단계마다 몇 건이 얼마나 바뀌었는지를 구체적으로 설명할 수 있다. 특히 완전 중복 행이 존재한다는 것은 데이터 수집 과정(예: 중복 적재, 재전송)에 문제가 있을 가능성을 시사하므로, 전처리뿐 아니라 데이터 소스 쪽에도 공유할 가치가 있는 정보라고 생각한다.

### 한계와 추가 확인 사항

지금 단계에서는 각 컬럼의 dtype과 숫자/날짜 변환 가능 여부를 아직 확인하지 않았다. price, quantity, unit_price 등이 실제로 숫자형으로 깨끗하게 저장되어 있는지는 다음 단계(숫자/날짜 변환)에서 확인할 예정이다.

## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [12]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\권혁수_PC\AppData\Local\Temp\ipykernel_55044\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\권혁수_PC\AppData\Local\Temp\ipykernel_55044\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/

In [13]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [14]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [15]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [16]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 2


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [17]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


## 2. 문자열·타입·결측 처리

- 적용한 문자열 정리: 모든 문자열(object) 컬럼에 대해 앞뒤 공백 제거(strip) 적용
- 숫자 변환 대상과 실패 건수: price 2건, quantity 5건, unit_price 2건 (errors='coerce'로 NaN 처리)
- 날짜 변환 대상과 실패 건수: order_date 3건, signup_date 2건 (errors='coerce'로 NaT 처리)
- 결측 처리 기준:
  - age(결측 4건) → 중앙값(41.0)으로 대체
  - city(결측 2건) → 'Unknown'으로 대체
  - order_status 표기 통일: completed/cancelled/refunded 외 12개 변형 (Complete, complete, COMPLETED, 완료, Cancel, cancel, CANCELLED, 취소, Refund, refund, REFUNDED, 환불)을 표준 3개 값으로 통일

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰

order_status 컬럼에는 표준 값(completed/cancelled/refunded) 외에도 대소문자가 다른 표기(Complete, COMPLETED 등)와 한글 표기(완료, 취소, 환불)가 섞여 있었다. status_map으로 치환한 결과 통일 전 15개였던 고유값이 통일 후 completed 180건, cancelled 63건, refunded 57건(합계 300건)의 3개 값으로 깔끔하게 정리되었다. 날짜 변환에서는 order_date 3건, signup_date 2건이 NaT로 변환되었고, 숫자 변환에서는 price 2건, quantity 5건, unit_price 2건이 NaN으로 변환되었다. 또한 order_date의 최댓값이 2028-12-31로, 현재 시점(2026년) 기준으로도 2년 이상 미래의 날짜가 포함되어 있다는 것을 확인했다.

### 나의 해석과 판단

order_status처럼 사람이 직접 입력했거나 여러 시스템에서 수집된
범주형 데이터는 표기가 다양하게 섞이기 쉽다고 판단했다. 이를 표준화하지 않고 그대로 집계하면 completed 주문이 15개 서로 다른 값으로 나뉘어 실제보다 과소 집계되는 문제가 발생했을 것이다. 날짜/숫자 변환 실패 건수를 errors='coerce'로 NaN/NaT 처리한 것은 변환이 안 되는 값을 무시하고 넘어가는 대신 명시적으로 '비어 있음'으로 표시해 이후 단계에서 놓치지 않고 확인할 수 있도록 하기 위함이다. 삭제가 아니라 변환 실패로 표시한 것이므로 아직 이 값들을 지울지 대체할지는 결정하지 않았고 이후 단계에서 업무 의미를 더 확인해야 한다고 판단했다.

### 업무·분석적 의미

표기가 통일되지 않은 범주값은 분석 결과를 눈에 띄지 않게 왜곡시킬 수 있다. 예를 들어 'completed'만 필터링하는 코드를 짰다면, 대문자나 한글로 적힌 완료 주문 5건이 조용히 누락되어 매출이 실제보다 적게 집계되었을 것이다. 이런 문제는 코드 자체는 에러 없이 실행되기 때문에 결과를 검증하지 않으면 발견하기 어렵다.

### 한계와 추가 확인 사항

변환 실패로 NaN/NaT가 된 값들(price 2건, quantity 5건, unit_price 2건, order_date 3건, signup_date 2건)을 어떻게 처리할지는 아직 결정하지 않았다. 또한 order_date 최댓값이 2028-12-31로 비정상적으로 먼 미래 날짜라는 점은 다음 단계(이상값 후보 확인)에서 추가로 들여다볼 필요가 있다.

## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [18]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [19]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


## 3. 중복·범주 표준화·이상값 후보

- 제거/보류한 중복: 4개 데이터셋 모두 완전 중복 행 2건씩 drop_duplicates()로 제거
- 표준화한 범주값: order_status 15개 표기(Complete, complete, COMPLETED, 완료 등)를 completed/cancelled/refunded 3개 값으로 통일
- 허용값 밖 값: 표준화 전 order_status에 12개의 비표준 표기가 존재했으나 표준화 후에는 허용값 밖 값 없음
- 이상값 후보:
  - products.price: 0 이하 2건 (최솟값 -20,000원, 음수 가격)
  - order_items.quantity: 0 이하 3건 (최솟값 -1, 음수 수량)
  - order_items.unit_price: 0 이하 1건
  - products.price 최댓값 9,999,999원 (다른 상품 가격대(중앙값 92,500원)에 비해 비정상적으로 높음, 더미/오류 값 의심)
  - order_items.quantity 최댓값 100 (평균 3.17, 75%가 4인 것에 비해 크게 벗어난 값)
  - orders.order_date 최댓값 2028-12-31 (현재 시점(2026-10) 기준 2년 이상 미래 날짜)

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰

price와 unit_price에서 음수 또는 0원인 값이 발견되었고, quantity에서도 음수(-1)와 비정상적으로 큰 값(100)이 함께 발견되었다. order_status는 표준화 전 15개의 서로 다른 표기(대소문자, 한글 혼용)가 존재했으나 표준화 후에는 3개 값으로 정리되어 허용값을 벗어나는 값이 남지 않았다.

### 나의 해석과 판단

이상값을 바로 삭제하지 않고 후보로만 검토해야 하는 이유는 음수 가격이나 음수 수량이 반드시 데이터 오류라고 단정할 수 없기 때문이다. 예를 들어 음수 수량은 반품 및 환불 처리 과정에서 발생한 레코드일 수 있고, 가격 0원은 이벤트성 무료 상품일 수 있다. 반면 price 9,999,999원이나 quantity 100처럼 극단적으로 튀는 값, 그리고 2028년이라는 미래 주문 날짜는 입력 오류나 테스트 더미 데이터일 가능성이 높다고 판단했다. 다만 이 역시 실제 업무 담당자 확인 없이 분석자 혼자 오류라고 단정하지 않았다. 이번 실습에서는 가이드에 따라 price/quantity/unit_price가 0 이하인 행만 분석 대상에서 제외하는 기준을 적용했다.

### 업무·분석적 의미

이상값을 아무 기준 없이 삭제하면 실제로는 의미 있는 비즈니스 이벤트 (반품, 프로모션 등)를 분석에서 놓칠 수 있다. 반대로 명백한 오류성 데이터(미래 날짜, 비정상적으로 큰 수량 등)를 그대로 두면 집계 결과가 왜곡된다. 따라서 이상값은 일괄 삭제가 아니라 후보로 분리해 검토하는 절차가 필요하며, 이 과정과 기준을 문서로 남겨야 추후 재현하거나 다른 담당자가 검증할 수 있다.

### 한계와 추가 확인 사항

이번 실습에서는 가이드가 제시한 단순 기준(0 이하 제외)만 적용했으며 price 9,999,999원이나 quantity 100, order_date 2028년처럼 '너무 큰' 값에 대한 별도의 상한 기준은 적용하지 않았다. 실제 업무에서는 이런 극단값들도 담당자 확인을 거쳐 별도 처리 기준을 정해야 한다.

### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [20]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [21]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [22]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 252140000.0


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [23]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


In [24]:
# product_id가 없어진 상품을 참조하는 경우 확인
removed_product_ids = set(products['product_id']) - set(products_clean['product_id'])
print('제거된 product_id들:', removed_product_ids)
print('이 상품들을 참조하는 order_items 건수:', 
      order_items_clean[order_items_clean['product_id'].isin(removed_product_ids)].shape[0])

# customer_id, order_id 쪽도 확인
removed_customer_ids = set(orders_clean['customer_id']) - set(customers_clean['customer_id'])
print('orders에는 있지만 customers에 없는 customer_id:', removed_customer_ids)

removed_order_ids = set(order_items_clean['order_id']) - set(orders_clean['order_id'])
print('order_items에는 있지만 orders에 없는 order_id:', removed_order_ids)

제거된 product_id들: {48, 73, 18, 95}
이 상품들을 참조하는 order_items 건수: 24
orders에는 있지만 customers에 없는 customer_id: {9992, 9991}
order_items에는 있지만 orders에 없는 order_id: {9992, 9991}


In [25]:
# 9991, 9992 주문의 실제 내용 확인
print(order_items_clean[order_items_clean['order_id'].isin([9991, 9992])])

     order_item_id  order_id  product_id  quantity  unit_price  line_total
149            150      9991          89       5.0    178000.0    890000.0
619            620      9992          98       1.0     32000.0     32000.0


## 4. PK/FK와 파생 컬럼 재검증

- 없는 `customer_id`: 2건 (orders에 customer_id 9991, 9992가 있으나 customers에는 없음)
- 없는 `order_id`: 2건 (order_items에 order_id 9991, 9992가 있으나 orders에는 없음)
- 없는 `product_id`: 26건 (이상값 처리로 제거된 product_id 18, 48, 73, 95를 참조하는 order_items 24건 + 그 외 2건은 위 orphan 주문(9991, 9992)과 겹침)
- 생성한 파생 컬럼: order_month, order_dayofweek (orders), line_total (order_items)
- `line_total` 검증: quantity × unit_price로 재계산한 head() 결과가 정상적으로 출력됨. 전처리 후 order_items의 line_total 합계는 252,140,000원임.

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰

전처리 후 관계를 재검증한 결과, 세 가지 체크 항목 모두 invalid_count가 0이 아니었다. 원인을 직접 추적한 결과, product_id 불일치(26건)의 대부분은 이상값 처리 단계에서 price가 0 이하라 제거했던 상품 4개(product_id 18, 48, 73, 95)를 참조하던 주문 상세 24건 때문이었다. customer_id와 order_id 불일치(각 2건)는 order_id 9991, 9992라는 두 건의 order_items 레코드가 원인이었는데, 이 두 주문은 orders 테이블에도 customers 테이블에도 대응하는 레코드가 전혀 없었다.

### 나의 해석과 판단

전처리 후 관계를 다시 검증해야 하는 이유는 전처리 과정에서 한쪽 테이블의 행을 삭제하면(이번 경우 price<=0 상품 제거) 다른 테이블에서 그 행을 참조하던 레코드가 '가리키는 대상이 사라진 참조'가 되기 때문이다. 만약 재검증을 하지 않았다면 products_clean에는 없는 상품을 order_items_clean이 계속 참조하는 상태로 다음 단계(집계, 분석)에 그대로 넘어갔을 것이고, products와 merge할 때 이 24건이 조용히 매칭 실패(left_only)로 빠지거나 결과가 왜곡되었을 것이다. customer_id/order_id 9991, 9992는 애초에 다른 테이블에 대응 레코드가 없는 상태로 원본에 포함되어 있었던 것으로 보이며, 전처리로 만들어진 문제가 아니라 원본 데이터 자체의 결함으로 판단했다.

### 업무·분석적 의미

PK/FK 관계가 깨진 상태로 분석을 진행하면, 매출이나 주문 건수 같은 핵심 지표가 조용히 줄어들거나(매칭 실패로 제외됨) 혹은 명시적으로 확인하지 않으면 원인 모를 오류로 이어질 수 있다. 특히 이번처럼 원본 데이터 자체에 존재하지 않는 고객/주문 ID를 참조하는 레코드가 있다는 것은 데이터 수집 시스템(주문 시스템과 고객 시스템 간 동기화 등)에 근본적인 문제가 있을 가능성을 시사하므로 업무 담당자에게 공유할 가치가 있는 발견이라고 판단했다.

## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [26]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,300,7
3,order_items,765,5,752,6


In [27]:
print('products price가 NaN인 행 수:', products_clean['price'].isna().sum())
print('order_items quantity가 NaN인 행 수:', order_items_clean['quantity'].isna().sum())
print('order_items unit_price가 NaN인 행 수:', order_items_clean['unit_price'].isna().sum())

products price가 NaN인 행 수: 0
order_items quantity가 NaN인 행 수: 0
order_items unit_price가 NaN인 행 수: 0


## 5. 전처리 전/후 비교

| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| customers 행 수 | 152 | 150 | 완전 중복 행 2건 제거 |
| products 행 수 | 102 | 96 | 완전 중복 2건 + price 변환 실패(NaN) 2건 + price ≤ 0 (음수) 2건 |
| orders 행 수 | 302 | 300 | 완전 중복 행 2건 제거 |
| order_items 행 수 | 765 | 752 | 완전 중복 2건 + quantity 변환 실패 5건 + unit_price 변환 실패 2건 + quantity ≤ 0 3건 + unit_price ≤ 0 1건 |
| customers 결측 | age 4건, city 2건 | 0건 | age는 중앙값(41.0), city는 'Unknown'으로 대체 |
| orders 컬럼 수 | 5개 | 7개 | order_month, order_dayofweek 파생 컬럼 추가 |
| order_items 컬럼 수 | 5개 | 6개 | line_total 파생 컬럼 추가 |
| 변환 실패(날짜) | order_date 3건, signup_date 2건 | 처리 보류 (NaT 유지) | 날짜는 수량/가격과 달리 자동 삭제하지 않음 |

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰

4개 데이터셋 모두 행 수가 줄었는데, 감소분을 세부 원인별로 모두 합산해보니 실제 감소량과 정확히 일치했다. 예를 들어 order_items는 13건이 줄었는데, 이는 완전 중복 2건, quantity/unit_price 숫자 변환 실패로 NaN이 된 7건, 그리고 quantity 또는 unit_price가 0 이하인 4건을 모두 더한 수치와 같았다. 숫자 변환 실패(NaN)가 이상값 필터링 (`> 0` 조건) 단계에서 자동으로 함께 제외된다는 것도 확인했다(NaN > 0은 False로 평가되기 때문임).

### 나의 해석과 판단

모든 행 수 변화의 원인을 구체적인 건수로 설명할 수 있다는 것은 전처리가 의도한 대로만 작동했고 예상치 못한 추가 손실이 없었다는 뜻이라고 판단했다. 특히 숫자 변환 실패(NaN)가 이상값 필터 조건에 딸려서 암묵적으로 제거된다는 점은 처음에는 의도한 설계가 아니었을 수 있지만, 결과적으로는 '변환 실패 = 분석에 사용할 수 없는 값'이라는 점에서 합리적인 결과로 이어졌다고 본다. 다만 이것이 우연히 맞아떨어진 것인지 의도된 설계인지는 코드를 명시적으로 분리해서 다시 확인할 필요가 있다고 생각한다.

### 업무·분석적 의미

행 수 변화를 "대략 몇 건 줄었다"가 아니라 원인별로 쪼개어 설명할 수 있어야 다른 사람이 전처리 결과를 신뢰하고 재현할 수 있다. 숫자가 설명되지 않고 남는 경우 데이터가 어디선가 더 유실되고 있다는 신호일 수 있으므로 이번처럼 변화량을 끝까지 검산하는 습관이 필요하다고 생각한다.

### 한계와 추가 확인 사항

이번 비교에서는 숫자 변환 실패와 이상값(≤0) 조건이 필터링 코드상 자연스럽게 함께 처리되었지만, 이 둘은 서로 다른 성격의 문제 (형식 오류 vs 비즈니스 규칙 위반)이므로 실제 업무에서는 각각 별도로 기록하고 처리 방식을 명시하는 것이 더 바람직하다고 생각한다.

## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [28]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('c:/dev/llm-data-analysis-study/llm-data-analysis-course/practice/chapter05/data/processed/customers_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/llm-data-analysis-course/practice/chapter05/data/processed/orders_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/llm-data-analysis-course/practice/chapter05/data/processed/order_items_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/llm-data-analysis-course/practice/chapter05/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [29]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: c:\dev\llm-data-analysis-study\llm-data-analysis-course\practice\chapter05\reports\ch05_preprocessing_summary.md


## 6. 재현 가능한 전처리 확인

- 생성된 clean CSV: customers_clean.csv, products_clean.csv, orders_clean.csv, order_items_clean.csv (노트북 실행 결과는 practice/chapter05/data/processed/에, 스크립트 실행 결과는 data/processed/에 각각 저장됨)
- 생성된 요약 보고서: ch05_preprocessing_summary.md (노트북/스크립트 각각의 REPORT_DIR 위치에 저장)
- `python scripts/preprocess_data.py` 재실행 결과: 정상 실행 완료. 전처리 전후 데이터 크기(customers 152→150, products 102→96, orders 302→300, order_items 765→752)와 관계 점검 결과(customer_id 2건, order_id 2건, product_id 26건 불일치)가 노트북에서 직접 처리했을 때와 정확히 동일하게 나옴
- 재실행 후 달라진 점: 수치상 차이 없음. 다만 스크립트는 입력을 practice/chapter05/data/raw로 지정했지만 출력은 프로젝트 공통 경로인 data/processed, reports로 저장하도록 설계되어 있어 노트북에서 저장한 위치(practice/chapter05/data/processed)와는 경로가 다름

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단

Notebook에서 한 번 성공하는 것과 스크립트로 재현 가능한 것은 코드가 "어쩌다 한 번 맞게 실행된 것"인지 "같은 입력에 대해 항상 같은 결과를 내는 것"인지의 차이라고 생각한다. 이번에 스크립트를 재실행했을 때 노트북에서 손으로 단계별로 처리했던 결과와 모든 수치(행 수 변화, 관계 점검 불일치 건수)가 정확히 일치했다는 것은 전처리 로직이 src/preprocessing.py에 함수로 잘 정리되어 있어 누가 언제 실행해도 동일한 결과를 재현할 수 있다는 것을 의미한다. 반면 노트북 셀을 수동으로 하나씩 실행하는 방식은 실행 순서를 건너뛰거나 중간에 변수를 잘못 덮어쓰는 등의 실수가 생기기 쉽다는 것을 이번 실습 중 직접 경험하기도 했다(노트북이 멈춰서 재시작 후 처음부터 다시 실행해야 했던 상황 등).

### 한계와 추가 확인 사항

스크립트의 출력 경로(data/processed, reports)가 노트북에서 저장한 경로(practice/chapter05/data/processed, practice/chapter05/reports)와 다르다는 점은 혼란을 줄 수 있어 실제 프로젝트였다면 두 출력 위치를 일치시키거나 스크립트 사용법 문서에 명확히 안내했어야 한다고 생각한다.

## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [30]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [31]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(96, 4)
(300, 7)
(752, 6)


C:\Users\권혁수_PC\AppData\Local\Temp\ipykernel_55044\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\권혁수_PC\AppData\Local\Temp\ipykernel_55044\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [32]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,752,6
2,orders,302,5,300,7
3,products,102,4,96,4


In [33]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [34]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 7. Chapter 05 최종 판단

### 내가 정한 전처리 원칙 3가지

1. 원본(raw)은 절대 덮어쓰지 않고, 항상 복사본(_clean)에서만 처리한다.
2. 숫자/날짜 변환은 errors='coerce'로 실패를 명시적으로 드러내고, 실패 건수를 반드시 확인한 뒤 다음 단계로 넘어간다.
3. 이상값과 관계 불일치는 발견 즉시 삭제하지 않고 먼저 원인(어떤 레코드가, 왜 그런 값을 갖게 됐는지)을 추적해 설명할 수 있을 때까지 보류한다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유

이상값 필터링(`price > 0`, `quantity > 0`, `unit_price > 0`) 단계가 가장 위험하다고 생각했다. 이 한 줄의 조건이 의도한 이상값(음수 가격 등)뿐 아니라, 숫자 변환에 실패해 NaN이 된 값들까지 조용히 함께 걸러냈기 때문이다(`NaN > 0`이 False로 평가됨). 의도와 결과가 우연히 합리적으로 맞아떨어졌지만, 만약 변환 실패 건수를 사전에 확인하지 않았다면 "이상값 몇 건을 제거했다"고만 보고하고, 실제로는 형식 오류로 인한 손실까지 섞여서 과소 보고했을 위험이 있었다.

### 다음 EDA에서 특히 주의할 데이터 특성

- order_items에 존재하지만 orders/customers에는 없는 고아 레코드 (order_id 9991, 9992)가 있었다는 것: EDA에서 merge를 다시 할 때도 동일한 매칭 실패가 재현되는지 확인이 필요하다.
- orders.order_date 최댓값이 2028-12-31로, 비정상적으로 먼 미래 날짜가 포함되어 있다는 것: 월별/연도별 집계 시 이 이상치 때문에 그래프나 트렌드가 왜곡될 수 있다.
- order_status가 원본에는 15개 변형으로 섞여 있었다는 것: 이번 전처리에서 표준화했지만, 이후 분석에서 다른 원본 파일을 추가로 쓸 경우 같은 문제가 재발할 수 있다.

### 현재 전처리의 한계

이번 전처리는 가이드가 제시한 단순 규칙(0 이하 제외, 중앙값/Unknown 대체)만 적용했으며, 고아 레코드(order_id 9991, 9992)는 발견만 하고 실제로 제거하거나 별도 처리하지는 않았다. 또한 price 9,999,999원 같은 극단값이나 quantity 100 같은 값은 '0 이하'라는 조건에 걸리지 않아 여전히 processed 데이터에 남아있을 가능성이 있다. 이런 항목들은 다음 EDA 단계에서 추가로 들여다봐야 한다.

## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


## 해당 코드의 문제점 검토

제안된 코드를 검토한 결과 다음과 같은 문제를 발견했다.

1. 원본 데이터를 직접 수정함
   `customers`, `orders`, `order_items`를 복사본(`.copy()`) 없이 바로 수정하고 있다. 이번 실습 내내 지켰던 'raw는 절대 덮어쓰지 않는다'는 원칙을 어기는 코드이다.

2. 결측치 처리 방식 — 평균(mean) vs 중앙값(median)
   나는 age 결측치를 중앙값으로 처리했는데, 이 코드는 평균을 쓰고 있다. 평균은 극단값(이상치)에 영향을 많이 받는다는 차이가 있으므로 둘 중 하나를 선택하려면 그 근거(데이터 분포, 극단값 존재 여부)를 먼저 확인해야 한다고 판단했다.

3. 날짜 변환 실패를 확인하지 않음
   `errors="coerce"`가 빠져 있어, 변환할 수 없는 값이 있으면 에러로 멈추거나 실패 여부를 전혀 알 수 없는 상태로 넘어갈 수 있다. 실제로 내 실습에서는 order_date 변환 실패가 3건 있었는데, 이 코드로는 그 사실 자체를 확인할 방법이 없다.

4. 중복 제거 기준 불충분
   `drop_duplicates()`만 있고 그 전에 몇 건이 중복인지 확인하는 과정이 없다. 나는 먼저 `duplicated().sum()`으로 2건임을 확인한 뒤 제거했는데, 이 코드는 몇 건이 사라졌는지 알 수 없는 상태로 바로 지우고 있다.

5. line_total 계산 전 숫자형 변환 누락
   quantity, unit_price가 문자열(쉼표 포함 등)일 가능성을 전혀 고려하지 않고 바로 곱셈을 하고 있다. 내 실습에서는 실제로 quantity 5건, unit_price 2건이 숫자 변환에 실패했는데, 이 코드라면 곱셈 자체가 에러로 멈추거나 의도하지 않은 결과를 낼 수 있다.


In [ ]:
# 더 안전한 수정코드

customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

# 결측치: 중앙값 사용, 처리 전 건수 확인
age_missing = customers_clean['age'].isna().sum()
age_median = customers_clean['age'].median()
customers_clean['age'] = customers_clean['age'].fillna(age_median)
print(f'age 결측치 {age_missing}건을 중앙값({age_median})으로 대체')

# 중복: 제거 전 건수 확인
dup_count = customers_clean.duplicated().sum()
customers_clean = customers_clean.drop_duplicates()
print(f'완전 중복 {dup_count}건 제거')

# 날짜: 변환 실패 확인
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())

# line_total: 숫자형 변환 후 계산
order_items_clean['quantity'] = pd.to_numeric(order_items_clean['quantity'], errors='coerce')
order_items_clean['unit_price'] = pd.to_numeric(order_items_clean['unit_price'], errors='coerce')
order_items_clean['line_total'] = order_items_clean['quantity'] * order_items_clean['unit_price']

## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [ ]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.


In [35]:
all_missing = pd.concat([
    missing_summary(customers_clean).reset_index().assign(dataset='customers'),
    missing_summary(products_clean).reset_index().assign(dataset='products'),
    missing_summary(orders_clean).reset_index().assign(dataset='orders'),
    missing_summary(order_items_clean).reset_index().assign(dataset='order_items'),
])
all_missing

,index,missing_count,missing_ratio,dataset
0,signup_date,2,1.33,customers
1,customer_id,0,0.00,customers
2,name,0,0.00,customers
3,gender,0,0.00,customers
4,age,0,0.00,customers
5,city,0,0.00,customers
0,product_id,0,0.00,products
1,product_name,0,0.00,products
2,category,0,0.00,products
3,price,0,0.00,products


In [36]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.


In [37]:
abnormal_age = customers_clean[(customers_clean['age'] < 18) | (customers_clean['age'] > 100)]
print('18세 미만 또는 100세 초과:', len(abnormal_age))
display(abnormal_age)

18세 미만 또는 100세 초과: 1


,customer_id,name,gender,age,city,signup_date
32,33,권승윤,F,150.0,고양,2025-09-14


In [ ]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.


In [38]:
monthly_order_count = orders_clean.groupby('order_month').size()
monthly_order_count

order_month
2025-07    19
2025-08    26
2025-09    12
2025-10    22
2025-11    23
2025-12    23
2026-01    26
2026-02    19
2026-03    22
2026-04    26
2026-05    20
2026-06    24
2026-07    12
2026-08    21
2027-01     1
2028-12     1
dtype: int64

In [ ]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.


In [39]:
order_items_clean.sort_values('line_total', ascending=False).head(10)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
750,751,295,31,100.0,155000.0,15500000.0
737,738,290,96,50.0,96000.0,4800000.0
319,320,129,23,5.0,197000.0,985000.0
9,10,5,65,5.0,193000.0,965000.0
24,25,11,65,5.0,193000.0,965000.0
293,294,119,36,5.0,193000.0,965000.0
116,117,51,65,5.0,193000.0,965000.0
20,21,9,36,5.0,193000.0,965000.0
290,291,117,47,5.0,188000.0,940000.0
491,492,195,99,5.0,187000.0,935000.0


# 과제 5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명

4개 데이터셋 모두 전처리 후 행 수가 줄었다.

- customers: 152 → 150 (완전 중복 행 2건 제거)
- products: 102 → 96 (완전 중복 2건 + price 숫자 변환 실패 2건 + price 0 이하 2건, 총 6건 감소)
- orders: 302 → 300 (완전 중복 행 2건 제거)
- order_items: 765 → 752 (완전 중복 2건 + quantity 변환 실패 5건 + unit_price 변환 실패 2건 + quantity 0 이하 3건 + unit_price 0 이하 1건, 총 13건 감소)

모든 데이터셋에서 공통적으로 작용한 요인은 완전 중복 행 제거였고, products와 order_items는 추가로 숫자 변환 실패와 이상값(0 이하) 조건이 겹쳐서 더 많이 감소했다. 감소분을 원인별로 모두 더했을 때 실제 감소량과 정확히 일치한다는 것을 확인했으므로 의도하지 않은 추가적인 데이터 손실은 없었다고 판단했다.

# 과제 6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트

다음과 같이 실제 값이 아닌 구조화된 요약 정보만 제공하는 프롬프트를 작성했다.

customers 데이터 결측치 현황
- 전체 행 수: 150
- age 결측: 4건 (전체의 2.63%)
- city 결측: 2건 (전체의 1.32%)
- 현재 age는 중앙값으로, city는 'Unknown' 문자열로 대체하는 방식을 검토 중이다.

아래 기준으로 이 처리 방식이 적절한지 검토해달라.
- age를 평균이 아닌 중앙값으로 대체한 것이 타당한지
- city를 삭제하지 않고 'Unknown' 범주로 남긴 것이 분석에 미칠 영향
- 두 컬럼의 결측 비율(2~3% 수준)을 고려할 때 대체 대신 해당 행을 제외하는 방법과 비교한 장단점
- 실제 업무에서 이 결측치 처리 전에 추가로 확인해야 할 사항

이 프롬프트는 실제 고객 이름이나 개인정보를 포함하지 않고 결측 건수/비율/처리 방식만 전달했다는 점에서, 노트북 섹션 23에서 안내한 '구조화된 정보만 제공' 원칙을 지키도록 작성했다.

## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


## 최종 제출 체크
- [x] raw 원본을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.